# Альфа-банк — Кредиты Наличными: РАСШИРЕННЫЙ pipeline для перехода с 91.65 → 92.0+

**Что добавлено относительно baseline:**
1. Расширенный listwise FE: delta-to-min/max/median, range, is_best, кроссы
2. Доменные фичи: monthly_payment, eva_minus_ncl, payment_to_income, rate*term
3. Count encoding + OOF target encoding по variant_no и его кроссам
4. XGBoost rank:pairwise как 3-я модель
5. lambdarank_truncation_level=8 (не 5)
6. label_gain=[0, 1] для бинарного таргета
7. Adversarial validation train vs test
8. Row-index leak audit
9. Ускорение CatBoost: metric_period=50, eval_metric=PairLogit

**Цель:** OOF NDCG@5 ~ 0.918-0.925.


In [1]:
!pip install -q lightgbm catboost xgboost optuna pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 25.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 40.2 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRanker, Pool
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
import gc, os, warnings, time
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)

DATA_DIR = './data'  # положи parquet'ы сюда
SEED = 42
N_SPLITS = 5

In [3]:
# Ячейка 2: скачиваем данные с Яндекс.Диска
import requests

YANDEX_LINK = "./data"

def download_yadisk(public_link, save_dir="./data"):
    os.makedirs(save_dir, exist_ok=True)
    api_url = "https://cloud-api.yandex.net/v1/disk/public/resources"
    
    # Получаем список файлов в папке
    r = requests.get(api_url, params={"public_key": public_link, "limit": 100})
    items = r.json().get("_embedded", {}).get("items", [])
    
    for item in items:
        name = item["name"]
        # Получаем прямую ссылку на скачивание
        dl = requests.get(
            "https://cloud-api.yandex.net/v1/disk/public/resources/download",
            params={"public_key": public_link, "path": "/" + name}
        ).json()
        href = dl["href"]
        print(f"Качаю {name}...")
        with requests.get(href, stream=True) as resp:
            with open(f"{save_dir}/{name}", "wb") as f:
                for chunk in resp.iter_content(8192):
                    f.write(chunk)
    print("Готово")

download_yadisk(YANDEX_LINK)
!ls -lh ./data/

Качаю commit.csv...
Качаю feature_description.csv...
Качаю features_small.pq...
Качаю model_credit_start.ipynb...
Качаю test_dataset_small.pq...
Качаю train_dataset_small.pq...
Готово
total 119M
-rw-r--r-- 1 root root  16M May 22 09:01 commit.csv
-rw-r--r-- 1 root root 2.2K May 22 09:01 feature_description.csv
-rw-r--r-- 1 root root  27M May 22 09:01 features_small.pq
-rw-r--r-- 1 root root 7.4K May 22 09:01 model_credit_start.ipynb
-rw-r--r-- 1 root root  32M May 22 09:01 test_dataset_small.pq
-rw-r--r-- 1 root root  45M May 22 09:01 train_dataset_small.pq


In [4]:
train = pd.read_parquet(f'{DATA_DIR}/train_dataset_small.pq')
test  = pd.read_parquet(f'{DATA_DIR}/test_dataset_small.pq')
feats = pd.read_parquet(f'{DATA_DIR}/features_small.pq')

print('train:', train.shape, '| test:', test.shape, '| feats:', feats.shape)
print('positive rate:', train['is_deal'].mean())

train: (845110, 23) | test: (936883, 22) | feats: (76583, 280)
positive rate: 0.04483558353350452


In [5]:
# Препроцессинг: строковые числа -> float
NUMERIC_FROM_STR = ['rate', 'eva', 'eva_perc', 'ncl']
for df in (train, test):
    for col in NUMERIC_FROM_STR:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype('float32')

# Категориальные строки -> codes
CAT_STR = ['verif_compl', 'offer_type', 'risk_level_map',
           'verif_need', 'need_2ndfl', 'channel', 'basket_name']

combined_cat_maps = {}
for col in CAT_STR:
    all_vals = pd.concat([train[col], test[col]], axis=0).fillna('NA').astype(str)
    cats = pd.Categorical(all_vals).categories
    combined_cat_maps[col] = {v: i for i, v in enumerate(cats)}
    train[col] = train[col].fillna('NA').astype(str).map(combined_cat_maps[col]).astype('int32')
    test[col]  = test[col].fillna('NA').astype(str).map(combined_cat_maps[col]).astype('int32')

print('Cat cardinalities:', {c: len(combined_cat_maps[c]) for c in CAT_STR})

Cat cardinalities: {'verif_compl': 2, 'offer_type': 2, 'risk_level_map': 4, 'verif_need': 2, 'need_2ndfl': 3, 'channel': 2, 'basket_name': 13}


In [6]:
# Препроцессинг features_small.pq
feats = feats.drop_duplicates(subset=['app_id', 'date_part'], keep='last').reset_index(drop=True)

obj_cols_feats = feats.select_dtypes(include=['object']).columns.tolist()
for col in obj_cols_feats:
    if col in ('app_id', 'date_part'): continue
    converted = pd.to_numeric(feats[col], errors='coerce')
    if converted.notna().sum() / max(feats[col].notna().sum(), 1) > 0.9:
        feats[col] = converted.astype('float32')
    else:
        feats[col] = pd.Categorical(feats[col].fillna('NA').astype(str)).codes.astype('int32')

for col in feats.select_dtypes(include=['object']).columns.tolist():
    if col in ('app_id', 'date_part'): continue
    feats[col] = pd.Categorical(feats[col].fillna('NA').astype(str)).codes.astype('int32')

print('feats dtypes:', feats.dtypes.value_counts().to_dict())

feats dtypes: {dtype('float64'): 200, dtype('int32'): 39, dtype('float32'): 39, dtype('O'): 2}


In [7]:
# JOIN
JOIN_KEYS = ['app_id', 'date_part']
feats_to_join = feats.drop(columns=[c for c in feats.columns
                                     if c in train.columns and c not in JOIN_KEYS],
                            errors='ignore')

train = train.merge(feats_to_join, on=JOIN_KEYS, how='left')
test  = test.merge(feats_to_join,  on=JOIN_KEYS, how='left')
print('train:', train.shape, '| test:', test.shape)

train: (845110, 301) | test: (936883, 300)


## БЛОК 1: Расширенный Listwise FE

Из Expedia ICDM 2013 5-place (arXiv:1311.7679): listwise + count features изолированно дали +2.38 п.п. NDCG@38.

In [8]:
LISTWISE_BASE = ['rate', 'eva', 'eva_perc', 'ncl', 'limit', 'term']

def add_listwise_features(df, group_col='request_id'):
    g = df.groupby(group_col)
    for col in LISTWISE_BASE:
        df[f'{col}_rank']    = g[col].rank(method='dense', pct=True).astype('float32')
        df[f'{col}_z']       = ((df[col] - g[col].transform('mean')) /
                                 g[col].transform('std').replace(0, 1)).astype('float32')

        g_min = g[col].transform('min')
        g_max = g[col].transform('max')
        g_med = g[col].transform('median')
        df[f'{col}_minus_min']    = (df[col] - g_min).astype('float32')
        df[f'{col}_minus_max']    = (df[col] - g_max).astype('float32')
        df[f'{col}_minus_median'] = (df[col] - g_med).astype('float32')
        df[f'{col}_to_min']    = (df[col] / g_min.replace(0, np.nan)).astype('float32')
        df[f'{col}_to_max']    = (df[col] / g_max.replace(0, np.nan)).astype('float32')
        df[f'{col}_range']     = (g_max - g_min).astype('float32')
        df[f'{col}_std']       = g[col].transform('std').astype('float32')

        df[f'is_min_{col}'] = (df[col] == g_min).astype('int8')
        df[f'is_max_{col}'] = (df[col] == g_max).astype('int8')

    df['n_variants_in_req'] = g[group_col].transform('size').astype('int32')
    return df

train = add_listwise_features(train)
test  = add_listwise_features(test)
print('После listwise FE: train', train.shape, '| test', test.shape)
gc.collect()

После listwise FE: train (845110, 368) | test (936883, 367)


0

In [9]:
def add_domain_features(df):
    df['eva_minus_ncl'] = (df['eva'] - df['ncl']).astype('float32')
    df['eva_div_ncl']   = (df['eva'] / (df['ncl'].abs() + 1e-9)).astype('float32')

    rate_m = df['rate'] / 12 / 100
    n = df['term'].astype('float32')
    df['monthly_payment'] = (df['limit'] * rate_m /
                              (1 - (1 + rate_m).pow(-n) + 1e-9)).astype('float32')
    df['total_interest']  = (df['monthly_payment'] * n - df['limit']).astype('float32')

    df['eva_div_limit'] = (df['eva'] / (df['limit'] + 1e-9)).astype('float32')

    df['limit_div_req']  = (df['limit'] / df['req_loan_amount'].replace(0, np.nan)).astype('float32')
    df['term_div_req']   = (df['term']  / df['req_term'].replace(0, np.nan)).astype('float32')
    df['limit_minus_req']= (df['limit'] - df['req_loan_amount']).astype('float32')
    df['term_minus_req'] = (df['term'] - df['req_term']).astype('float32')

    df['rate_times_term'] = (df['rate'] * df['term']).astype('float32')

    if 'profit_income_out_rur_amt_cm' in df.columns:
        df['payment_to_income'] = (df['monthly_payment'] /
                                    (df['profit_income_out_rur_amt_cm'].abs() + 1e-9)
                                  ).astype('float32')
    if 'age' in df.columns:
        df['term_to_age_left'] = (df['term'] / (12 * (70 - df['age']).clip(1, 70))).astype('float32')

    return df

train = add_domain_features(train)
test  = add_domain_features(test)

DOMAIN_FEATS = ['eva_minus_ncl', 'monthly_payment', 'eva_div_limit',
                'limit_div_req', 'rate_times_term']

for col in DOMAIN_FEATS:
    for df in (train, test):
        g = df.groupby('request_id')[col]
        df[f'{col}_rank'] = g.rank(method='dense', pct=True).astype('float32')

print('После domain FE: train', train.shape, '| test', test.shape)
gc.collect()

После domain FE: train (845110, 385) | test (936883, 384)


0

## БЛОК 2: Count Encoding и OOF Target Encoding

In [10]:
COUNT_COLS = ['variant_no', 'basket_name', 'offer_type', 'risk_level_map', 'offer_id']

all_df = pd.concat([train[COUNT_COLS], test[COUNT_COLS]], axis=0)
for col in COUNT_COLS:
    cnt_map = all_df[col].value_counts().to_dict()
    train[f'{col}_cnt'] = train[col].map(cnt_map).astype('float32')
    test[f'{col}_cnt']  = test[col].map(cnt_map).astype('float32')

for c1, c2 in [('variant_no', 'offer_type'), ('variant_no', 'risk_level_map')]:
    key_train = train[c1].astype(str) + '_' + train[c2].astype(str)
    key_test  = test[c1].astype(str)  + '_' + test[c2].astype(str)
    key_all = pd.concat([key_train, key_test], axis=0)
    cnt = key_all.value_counts().to_dict()
    train[f'{c1}_{c2}_cnt'] = key_train.map(cnt).astype('float32')
    test[f'{c1}_{c2}_cnt']  = key_test.map(cnt).astype('float32')

del all_df; gc.collect()
print('Count encoded')

Count encoded


In [11]:
# OOF Target Encoding (GroupKFold по request_id, Bayesian smoothing)
TE_COLS = ['variant_no', 'basket_name', 'offer_type', 'risk_level_map', 'offer_id']
TE_CROSSES = []
if 'clientsegment' in train.columns:
    train['variant_x_segment'] = (train['variant_no'].astype(str) + '_' +
                                    train['clientsegment'].astype(str))
    test['variant_x_segment']  = (test['variant_no'].astype(str) + '_' +
                                    test['clientsegment'].astype(str))
    TE_CROSSES.append('variant_x_segment')

train['variant_x_offer_type'] = (train['variant_no'].astype(str) + '_' +
                                   train['offer_type'].astype(str))
test['variant_x_offer_type']  = (test['variant_no'].astype(str) + '_' +
                                   test['offer_type'].astype(str))
TE_CROSSES.append('variant_x_offer_type')

TE_ALL = TE_COLS + TE_CROSSES

train = train.sort_values('request_id').reset_index(drop=True)
test  = test.sort_values('request_id').reset_index(drop=True)

g_train = train['request_id'].values
y_train_arr = train['is_deal'].values
GLOBAL_MEAN = float(y_train_arr.mean())
SMOOTHING = 20.0

def oof_target_encode(train_df, test_df, col, y, groups, smoothing=20.0):
    oof = np.full(len(train_df), GLOBAL_MEAN, dtype='float32')
    gkf = GroupKFold(n_splits=5)
    for tr_idx, va_idx in gkf.split(train_df, y, groups=groups):
        stats_df = pd.DataFrame({col: train_df[col].iloc[tr_idx].values, 'y': y[tr_idx]})
        stats = stats_df.groupby(col)['y'].agg(['mean', 'count'])
        stats['te'] = (stats['mean'] * stats['count'] + GLOBAL_MEAN * smoothing) / (stats['count'] + smoothing)
        mapping = stats['te'].to_dict()
        oof[va_idx] = train_df[col].iloc[va_idx].map(mapping).fillna(GLOBAL_MEAN).values

    full_stats_df = pd.DataFrame({col: train_df[col].values, 'y': y})
    full_stats = full_stats_df.groupby(col)['y'].agg(['mean', 'count'])
    full_stats['te'] = (full_stats['mean'] * full_stats['count'] + GLOBAL_MEAN * smoothing) / (full_stats['count'] + smoothing)
    test_enc = test_df[col].map(full_stats['te'].to_dict()).fillna(GLOBAL_MEAN).astype('float32').values
    return oof, test_enc

for col in TE_ALL:
    print(f'TE: {col}...', end=' ')
    oof, te = oof_target_encode(train, test, col, y_train_arr, g_train, SMOOTHING)
    train[f'{col}_te'] = oof
    test[f'{col}_te']  = te
    print(f'mean={oof.mean():.4f}')

for col in TE_CROSSES:
    train.drop(columns=[col], inplace=True)
    test.drop(columns=[col], inplace=True)

print('TE done')
gc.collect()

TE: variant_no... mean=0.0449
TE: basket_name... mean=0.0447
TE: offer_type... mean=0.0448
TE: risk_level_map... mean=0.0448
TE: offer_id... mean=0.0450
TE: variant_x_segment... mean=0.0450
TE: variant_x_offer_type... mean=0.0449
TE done


0

## БЛОК 3: Row-index leak audit

In [12]:
audit_df = train.copy()
audit_df['row_pos_in_req'] = audit_df.groupby('request_id').cumcount()
corr_row = audit_df.groupby('row_pos_in_req')['is_deal'].mean()
print('mean is_deal by row_pos_in_req (первые 15 позиций):')
print(corr_row.head(15).round(4))
print()
print('Если одна позиция явно чаще даёт 1 — возможен leak.')
del audit_df; gc.collect()

mean is_deal by row_pos_in_req (первые 15 позиций):
row_pos_in_req
0     0.0964
1     0.0776
2     0.0683
3     0.0580
4     0.0568
5     0.0505
6     0.0505
7     0.0484
8     0.0453
9     0.0461
10    0.0421
11    0.0420
12    0.0401
13    0.0393
14    0.0380
Name: is_deal, dtype: float64

Если одна позиция явно чаще даёт 1 — возможен leak.


0

## БЛОК 4: Adversarial Validation train vs test

In [13]:
adv_train = train.copy()
adv_test = test.copy()
adv_train['__is_test'] = 0
adv_test['__is_test'] = 1
adv = pd.concat([adv_train, adv_test], axis=0, ignore_index=True)

adv_features = [c for c in adv.columns
                if c not in ('app_id', 'request_received', 'date_part', 'request_id',
                              'offer_id', 'is_deal', '__is_test')
                and adv[c].dtype != 'object']

from sklearn.model_selection import train_test_split
X_adv, y_adv = adv[adv_features].fillna(-1), adv['__is_test']
X_adv_tr, X_adv_va, y_adv_tr, y_adv_va = train_test_split(X_adv, y_adv, test_size=0.2, random_state=SEED)

adv_model = lgb.LGBMClassifier(n_estimators=200, learning_rate=0.05, num_leaves=63, verbose=-1)
adv_model.fit(X_adv_tr, y_adv_tr, eval_set=[(X_adv_va, y_adv_va)], callbacks=[lgb.early_stopping(20, verbose=False)])
auc = roc_auc_score(y_adv_va, adv_model.predict_proba(X_adv_va)[:, 1])
print(f'Adversarial AUC: {auc:.4f}')
if auc > 0.65:
    print('Сильный drift между train и test')
    imp = pd.DataFrame({'feat': adv_features,
                         'imp': adv_model.feature_importances_}).sort_values('imp', ascending=False)
    print('Топ-10 дрейфующих фичей:')
    print(imp.head(10))
else:
    print('Drift умеренный')
del adv, adv_train, adv_test, X_adv, X_adv_tr, X_adv_va
gc.collect()

Adversarial AUC: 1.0000
Сильный drift между train и test
Топ-10 дрейфующих фичей:
                feat   imp
0    req_loan_amount  2508
387   basket_name_te  2058
388    offer_type_te  1557
16       basket_name   646
8              limit   302
2        verif_compl   292
10               eva   262
380  basket_name_cnt   237
3         variant_no   215
11          eva_perc   158


23

## БЛОК 5: Финальные фичи и три модели

In [14]:
DROP_COLS = ['app_id', 'request_received', 'date_part', 'request_id',
             'offer_id', 'is_deal', 'nb_date', '__is_test']

feature_cols = [c for c in train.columns
                if c not in DROP_COLS and c in test.columns]
print('Финальное число фичей:', len(feature_cols))

CAT_FEATURES = ['variant_no', 'verif_compl', 'offer_type', 'risk_level_map',
                'verif_need', 'need_2ndfl', 'channel', 'basket_name', 'pil1mtrx_offer']
CAT_FEATURES = [c for c in CAT_FEATURES if c in feature_cols]
print('cat features:', CAT_FEATURES)

Финальное число фичей: 392
cat features: ['variant_no', 'verif_compl', 'offer_type', 'risk_level_map', 'verif_need', 'need_2ndfl', 'channel', 'basket_name', 'pil1mtrx_offer']


In [15]:
train = train.sort_values('request_id').reset_index(drop=True)
test  = test.sort_values('request_id').reset_index(drop=True)

X_train = train[feature_cols].copy()
y_train_arr = train['is_deal'].values
g_train = train['request_id'].values
X_test = test[feature_cols].copy()

for c in CAT_FEATURES:
    X_train[c] = X_train[c].fillna(-1).astype('int32')
    X_test[c]  = X_test[c].fillna(-1).astype('int32')

print('X_train:', X_train.shape, '| X_test:', X_test.shape)

X_train: (845110, 392) | X_test: (936883, 392)


In [16]:
def ndcg_at_k_grouped(df, score_col, target_col='is_deal', group_col='request_id', k=5):
    scores = []
    for req, g in df.groupby(group_col, sort=False):
        if g[target_col].sum() == 0:
            continue
        order = np.argsort(-g[score_col].values)
        ranks = np.where(g[target_col].values[order] == 1)[0]
        if len(ranks) == 0 or ranks[0] >= k:
            scores.append(0.0)
        else:
            scores.append(1.0 / np.log2(ranks[0] + 2))
    return np.mean(scores)

### Модель 1: LightGBM lambdarank (truncation=8, label_gain=[0,1])

In [17]:
gkf = GroupKFold(n_splits=N_SPLITS)
oof_lgb = np.zeros(len(train))
test_pred_lgb = np.zeros(len(test))

for fold, (tr_idx, va_idx) in enumerate(gkf.split(X_train, y_train_arr, groups=g_train)):
    print(f'\n===== LGB Fold {fold} =====')

    tr_groups = train.iloc[tr_idx].groupby('request_id', sort=False).size().values
    va_groups = train.iloc[va_idx].groupby('request_id', sort=False).size().values

    dtr = lgb.Dataset(X_train.iloc[tr_idx], y_train_arr[tr_idx], group=tr_groups,
                      categorical_feature=CAT_FEATURES)
    dva = lgb.Dataset(X_train.iloc[va_idx], y_train_arr[va_idx], group=va_groups,
                      categorical_feature=CAT_FEATURES, reference=dtr)

    params = {
        'objective': 'lambdarank',
        'metric': 'ndcg',
        'eval_at': [5],
        'lambdarank_truncation_level': 8,
        'lambdarank_norm': True,
        'label_gain': [0, 1],
        'learning_rate': 0.03,
        'num_leaves': 127,
        'min_data_in_leaf': 100,
        'feature_fraction': 0.8,
        'feature_fraction_bynode': 0.8,
        'bagging_fraction': 0.8,
        'bagging_freq': 5,
        'lambda_l1': 0.1,
        'lambda_l2': 1.0,
        'verbose': -1,
        'seed': SEED + fold,
    }

    model = lgb.train(params, dtr, num_boost_round=5000, valid_sets=[dva],
                      callbacks=[lgb.early_stopping(200), lgb.log_evaluation(200)])

    oof_lgb[va_idx] = model.predict(X_train.iloc[va_idx])
    test_pred_lgb += model.predict(X_test) / N_SPLITS
    gc.collect()

train['oof_lgb'] = oof_lgb
print(f'\nOOF NDCG@5 (LGB): {ndcg_at_k_grouped(train, "oof_lgb"):.5f}')


===== LGB Fold 0 =====
Training until validation scores don't improve for 200 rounds
[200]	valid_0's ndcg@5: 0.917606
[400]	valid_0's ndcg@5: 0.916894
Early stopping, best iteration is:
[248]	valid_0's ndcg@5: 0.918089

===== LGB Fold 1 =====
Training until validation scores don't improve for 200 rounds
[200]	valid_0's ndcg@5: 0.909077
[400]	valid_0's ndcg@5: 0.908706
Early stopping, best iteration is:
[299]	valid_0's ndcg@5: 0.910147

===== LGB Fold 2 =====
Training until validation scores don't improve for 200 rounds
[200]	valid_0's ndcg@5: 0.913697
[400]	valid_0's ndcg@5: 0.915095
Early stopping, best iteration is:
[394]	valid_0's ndcg@5: 0.915248

===== LGB Fold 3 =====
Training until validation scores don't improve for 200 rounds
[200]	valid_0's ndcg@5: 0.911311
[400]	valid_0's ndcg@5: 0.91035
Early stopping, best iteration is:
[249]	valid_0's ndcg@5: 0.912293

===== LGB Fold 4 =====
Training until validation scores don't improve for 200 rounds
[200]	valid_0's ndcg@5: 0.915166
[4

### Модель 2: CatBoost YetiRankPairwise (УСКОРЕННАЯ)

**Главные ускорители:**
- `metric_period=50` (по умолчанию 5) — основная причина 40 мин/фолд была bouncing GPU↔CPU из-за частой проверки NDCG
- `eval_metric='PairLogit'` — считается на GPU (NDCG на GPU не реализован)
- `max_ctr_complexity=2` — меньше CTR-комбинаций

Цель: 5-10 мин/фолд.

In [18]:
cb_cat_features = [feature_cols.index(c) for c in CAT_FEATURES]
oof_cb = np.zeros(len(train))
test_pred_cb = np.zeros(len(test))

for fold, (tr_idx, va_idx) in enumerate(gkf.split(X_train, y_train_arr, groups=g_train)):
    print(f'\n===== CB Fold {fold} =====')
    t0 = time.time()

    tr_pool = Pool(X_train.iloc[tr_idx], y_train_arr[tr_idx],
                   group_id=g_train[tr_idx],
                   cat_features=cb_cat_features)
    va_pool = Pool(X_train.iloc[va_idx], y_train_arr[va_idx],
                   group_id=g_train[va_idx],
                   cat_features=cb_cat_features)

    model_cb = CatBoostRanker(
        loss_function='YetiRankPairwise',
        eval_metric='NDCG:top=5;type=Exp',
        metric_period=200,
        iterations=2500,
        learning_rate=0.05,
        depth=8,
        l2_leaf_reg=3.0,
        max_ctr_complexity=2,
        random_seed=SEED + fold,
        verbose=200,
        task_type='GPU',
        devices='0',
        early_stopping_rounds=400,
    )

    model_cb.fit(tr_pool, eval_set=va_pool)

    oof_cb[va_idx] = model_cb.predict(va_pool)
    test_pred_cb += model_cb.predict(Pool(X_test, cat_features=cb_cat_features)) / N_SPLITS

    print(f'Fold {fold} time: {(time.time() - t0)/60:.1f} мин, best iter: {model_cb.best_iteration_}')
    gc.collect()

train['oof_cb'] = oof_cb
print(f'\nOOF NDCG@5 (CB): {ndcg_at_k_grouped(train, "oof_cb"):.5f}')


===== CB Fold 0 =====
Groupwise loss function. OneHotMaxSize set to 10


Metric PFound is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric NDCG:top=5;type=Exp is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	test: 0.8709137	best: 0.8709137 (0)	total: 1.16s	remaining: 48m 24s
200:	test: 0.9159526	best: 0.9160027 (196)	total: 3m 59s	remaining: 45m 40s
400:	test: 0.9175662	best: 0.9178246 (376)	total: 7m 58s	remaining: 41m 45s
600:	test: 0.9177101	best: 0.9179309 (404)	total: 11m 57s	remaining: 37m 48s
800:	test: 0.9180466	best: 0.9182060 (753)	total: 15m 56s	remaining: 33m 49s
1000:	test: 0.9183079	best: 0.9188118 (970)	total: 19m 56s	remaining: 29m 51s
1200:	test: 0.9183787	best: 0.9188118 (970)	total: 23m 55s	remaining: 25m 52s
bestTest = 0.9188118372
bestIteration = 970
Shrink model to first 971 iterations.
Fold 0 time: 27.4 мин, best iter: 970

===== CB Fold 1 =====
Groupwise loss function. OneHotMaxSize set to 10


Metric PFound is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric NDCG:top=5;type=Exp is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	test: 0.8608293	best: 0.8608293 (0)	total: 1.19s	remaining: 49m 25s
200:	test: 0.9054164	best: 0.9054164 (200)	total: 4m 2s	remaining: 46m 14s
400:	test: 0.9077806	best: 0.9077806 (400)	total: 8m 2s	remaining: 42m 5s
600:	test: 0.9083060	best: 0.9085682 (579)	total: 12m 2s	remaining: 38m 3s
800:	test: 0.9086986	best: 0.9087695 (765)	total: 16m 2s	remaining: 34m
1000:	test: 0.9092814	best: 0.9094552 (974)	total: 20m 2s	remaining: 30m
1200:	test: 0.9096418	best: 0.9097115 (1190)	total: 24m 2s	remaining: 26m
1400:	test: 0.9100099	best: 0.9102761 (1389)	total: 28m 3s	remaining: 22m
1600:	test: 0.9103465	best: 0.9103670 (1537)	total: 32m 3s	remaining: 18m
1800:	test: 0.9102043	best: 0.9105410 (1621)	total: 36m 4s	remaining: 13m 59s
2000:	test: 0.9103135	best: 0.9106023 (1990)	total: 40m 4s	remaining: 9m 59s
2200:	test: 0.9104107	best: 0.9106929 (2021)	total: 44m 5s	remaining: 5m 59s
2400:	test: 0.9101119	best: 0.9106929 (2021)	total: 48m 5s	remaining: 1m 58s
bestTest = 0.9106928711
bestI

...

0:	test: 0.8658644	best: 0.8658644 (0)	total: 1.19s	remaining: 49m 39s
200:	test: 0.9113956	best: 0.9116018 (196)	total: 4m 2s	remaining: 46m 11s
400:	test: 0.9135685	best: 0.9135685 (398)	total: 8m 2s	remaining: 42m 4s
600:	test: 0.9144222	best: 0.9147304 (591)	total: 12m 1s	remaining: 37m 59s
800:	test: 0.9154399	best: 0.9155950 (777)	total: 16m	remaining: 33m 58s
1000:	test: 0.9153797	best: 0.9156172 (831)	total: 20m	remaining: 29m 58s
1200:	test: 0.9159694	best: 0.9161721 (1105)	total: 24m	remaining: 25m 58s
1400:	test: 0.9159582	best: 0.9162553 (1257)	total: 28m	remaining: 21m 58s
1600:	test: 0.9160963	best: 0.9165656 (1523)	total: 32m	remaining: 17m 58s
1800:	test: 0.9162747	best: 0.9165656 (1523)	total: 36m 1s	remaining: 13m 58s
bestTest = 0.9165656095
bestIteration = 1523
Shrink model to first 1524 iterations.
Fold 2 time: 38.6 мин, best iter: 1523

===== CB Fold 3 =====
Groupwise loss function. OneHotMaxSize set to 10


Metric PFound is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric NDCG:top=5;type=Exp is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	test: 0.8721943	best: 0.8721943 (0)	total: 1.18s	remaining: 49m 9s
200:	test: 0.9100140	best: 0.9102208 (191)	total: 4m 2s	remaining: 46m 12s
400:	test: 0.9117156	best: 0.9117444 (397)	total: 8m 2s	remaining: 42m 3s
600:	test: 0.9121023	best: 0.9122505 (523)	total: 12m 1s	remaining: 38m 1s
800:	test: 0.9118973	best: 0.9124490 (679)	total: 16m 1s	remaining: 33m 59s
1000:	test: 0.9129780	best: 0.9131270 (987)	total: 20m 1s	remaining: 29m 58s
1200:	test: 0.9128345	best: 0.9132495 (1094)	total: 24m	remaining: 25m 58s
1400:	test: 0.9132205	best: 0.9133164 (1223)	total: 28m	remaining: 21m 58s
1600:	test: 0.9139740	best: 0.9139894 (1598)	total: 31m 59s	remaining: 17m 58s
1800:	test: 0.9138408	best: 0.9141151 (1644)	total: 35m 59s	remaining: 13m 58s
2000:	test: 0.9136373	best: 0.9141151 (1644)	total: 39m 59s	remaining: 9m 58s
bestTest = 0.9141151373
bestIteration = 1644
Shrink model to first 1645 iterations.
Fold 3 time: 41.0 мин, best iter: 1644

===== CB Fold 4 =====
Groupwise loss functi

Metric PFound is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric NDCG:top=5;type=Exp is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	test: 0.8685477	best: 0.8685477 (0)	total: 1.18s	remaining: 49m 9s
200:	test: 0.9100096	best: 0.9102226 (198)	total: 4m 1s	remaining: 46m 6s
400:	test: 0.9118969	best: 0.9123083 (369)	total: 8m 1s	remaining: 41m 57s
600:	test: 0.9132918	best: 0.9135282 (579)	total: 12m	remaining: 37m 57s
800:	test: 0.9136926	best: 0.9138683 (768)	total: 16m	remaining: 33m 56s
1000:	test: 0.9138797	best: 0.9142510 (952)	total: 19m 59s	remaining: 29m 56s
1200:	test: 0.9141618	best: 0.9145954 (1084)	total: 23m 59s	remaining: 25m 56s
1400:	test: 0.9143319	best: 0.9146490 (1299)	total: 27m 58s	remaining: 21m 56s
1600:	test: 0.9140463	best: 0.9146490 (1299)	total: 31m 58s	remaining: 17m 57s
bestTest = 0.9146489854
bestIteration = 1299
Shrink model to first 1300 iterations.
Fold 4 time: 34.1 мин, best iter: 1299

OOF NDCG@5 (CB): 0.91497


In [19]:
print(2)

2


In [20]:
# === ПРОДОЛЖЕНИЕ CatBoost CV после перезапуска VS Code ===
# Ядро Colab живо: oof_cb / test_pred_cb уже содержат посчитанные фолды (0,1,2).
# Эта ячейка определяет, какие фолды осталось обучить, и доводит CV до конца,
# НЕ пересчитывая готовые фолды и не задваивая их вклад в test_pred_cb.
import time, gc
import numpy as np
from catboost import Pool, CatBoostRanker

# splits детерминированы (тот же gkf) -> индексы фолдов совпадут с прошлым прогоном
splits = list(gkf.split(X_train, y_train_arr, groups=g_train))

# подстраховка, если каких-то переменных нет в памяти
if 'cb_cat_features' not in globals():
    cb_cat_features = [feature_cols.index(c) for c in CAT_FEATURES]
if 'oof_cb' not in globals():
    oof_cb = np.zeros(len(train))
if 'test_pred_cb' not in globals():
    test_pred_cb = np.zeros(len(test))

# фолд считаем готовым, если ВСЕ его valid-позиции в oof_cb уже заполнены
# (скоры CatBoost практически никогда не равны ровно 0)
done_folds = [f for f, (_, va) in enumerate(splits) if np.all(oof_cb[va] != 0)]
todo_folds = [f for f in range(N_SPLITS) if f not in done_folds]
print('Готовы фолды:', done_folds, '| Осталось обучить:', todo_folds)

for fold in todo_folds:
    tr_idx, va_idx = splits[fold]
    print(f'\n===== CB Fold {fold} (resume) =====')
    t0 = time.time()

    tr_pool = Pool(X_train.iloc[tr_idx], y_train_arr[tr_idx],
                   group_id=g_train[tr_idx],
                   cat_features=cb_cat_features)
    va_pool = Pool(X_train.iloc[va_idx], y_train_arr[va_idx],
                   group_id=g_train[va_idx],
                   cat_features=cb_cat_features)

    model_cb = CatBoostRanker(
        loss_function='YetiRankPairwise',
        eval_metric='NDCG:top=5;type=Exp',
        metric_period=200,
        iterations=2500,
        learning_rate=0.05,
        depth=8,
        l2_leaf_reg=3.0,
        max_ctr_complexity=2,
        random_seed=SEED + fold,
        verbose=200,
        task_type='GPU',
        devices='0',
        early_stopping_rounds=400,
    )

    model_cb.fit(tr_pool, eval_set=va_pool)

    oof_cb[va_idx] = model_cb.predict(va_pool)
    test_pred_cb += model_cb.predict(Pool(X_test, cat_features=cb_cat_features)) / N_SPLITS

    print(f'Fold {fold} time: {(time.time() - t0)/60:.1f} мин, best iter: {model_cb.best_iteration_}')
    gc.collect()

train['oof_cb'] = oof_cb
print(f'\nOOF NDCG@5 (CB): {ndcg_at_k_grouped(train, "oof_cb"):.5f}')


Готовы фолды: [0, 1, 2, 3, 4] | Осталось обучить: []

OOF NDCG@5 (CB): 0.91497


### Модель 3: XGBoost rank:pairwise

In [21]:
oof_xgb = np.zeros(len(train))
test_pred_xgb = np.zeros(len(test))

X_train_xgb = X_train.fillna(-999.0)
X_test_xgb  = X_test.fillna(-999.0)

for fold, (tr_idx, va_idx) in enumerate(gkf.split(X_train_xgb, y_train_arr, groups=g_train)):
    print(f'\n===== XGB Fold {fold} =====')
    t0 = time.time()

    tr_groups = train.iloc[tr_idx].groupby('request_id', sort=False).size().values
    va_groups = train.iloc[va_idx].groupby('request_id', sort=False).size().values

    dtr = xgb.DMatrix(X_train_xgb.iloc[tr_idx], label=y_train_arr[tr_idx], group=tr_groups)
    dva = xgb.DMatrix(X_train_xgb.iloc[va_idx], label=y_train_arr[va_idx], group=va_groups)

    params = {
        'objective': 'rank:pairwise',
        'eval_metric': 'ndcg@5',
        'learning_rate': 0.05,
        'max_depth': 8,
        'min_child_weight': 5,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'reg_lambda': 1.0,
        'tree_method': 'hist',
        'device': 'cuda',
        'seed': SEED + fold,
        'verbosity': 0,
    }

    model_xgb = xgb.train(params, dtr, num_boost_round=3000,
                           evals=[(dva, 'va')],
                           early_stopping_rounds=200,
                           verbose_eval=200)

    oof_xgb[va_idx] = model_xgb.predict(dva)
    test_pred_xgb += model_xgb.predict(xgb.DMatrix(X_test_xgb)) / N_SPLITS

    print(f'XGB Fold {fold} time: {(time.time() - t0)/60:.1f} мин')
    gc.collect()

train['oof_xgb'] = oof_xgb
print(f'\nOOF NDCG@5 (XGB): {ndcg_at_k_grouped(train, "oof_xgb"):.5f}')


===== XGB Fold 0 =====
[0]	va-ndcg@5:0.88853
[200]	va-ndcg@5:0.91705
[400]	va-ndcg@5:0.91875
[600]	va-ndcg@5:0.91817
[668]	va-ndcg@5:0.91842
XGB Fold 0 time: 0.7 мин

===== XGB Fold 1 =====
[0]	va-ndcg@5:0.88016
[200]	va-ndcg@5:0.91013
[400]	va-ndcg@5:0.91087
[570]	va-ndcg@5:0.91049
XGB Fold 1 time: 0.6 мин

===== XGB Fold 2 =====
[0]	va-ndcg@5:0.88552
[200]	va-ndcg@5:0.91389
[400]	va-ndcg@5:0.91588
[600]	va-ndcg@5:0.91634
[665]	va-ndcg@5:0.91557
XGB Fold 2 time: 0.7 мин

===== XGB Fold 3 =====
[0]	va-ndcg@5:0.88115
[200]	va-ndcg@5:0.91254
[400]	va-ndcg@5:0.91332
[542]	va-ndcg@5:0.91200
XGB Fold 3 time: 0.6 мин

===== XGB Fold 4 =====
[0]	va-ndcg@5:0.88811
[200]	va-ndcg@5:0.91522
[400]	va-ndcg@5:0.91585
[600]	va-ndcg@5:0.91454
[605]	va-ndcg@5:0.91431
XGB Fold 4 time: 0.6 мин

OOF NDCG@5 (XGB): 0.91416


### Ансамбль 3 моделей по рангам

In [22]:
def rank_within_group(df, score_col, group_col='request_id'):
    return df.groupby(group_col)[score_col].rank(method='average', pct=True).values

train['r_lgb'] = rank_within_group(train, 'oof_lgb')
train['r_cb']  = rank_within_group(train, 'oof_cb')
train['r_xgb'] = rank_within_group(train, 'oof_xgb')

print('Подбор весов:')
best_score, best_w = 0, (0.34, 0.33, 0.33)
for w_lgb in np.arange(0, 1.01, 0.05):
    for w_cb in np.arange(0, 1.01 - w_lgb, 0.05):
        w_xgb = 1.0 - w_lgb - w_cb
        if w_xgb < 0: continue
        train['blend'] = w_lgb * train['r_lgb'] + w_cb * train['r_cb'] + w_xgb * train['r_xgb']
        s = ndcg_at_k_grouped(train, 'blend')
        if s > best_score:
            best_score, best_w = s, (w_lgb, w_cb, w_xgb)

print(f'Лучшие веса LGB/CB/XGB: {best_w[0]:.2f} / {best_w[1]:.2f} / {best_w[2]:.2f}')
print(f'OOF NDCG@5: {best_score:.5f}')
print(f'Балл: {best_score * 100:.3f}')

Подбор весов:
Лучшие веса LGB/CB/XGB: 0.20 / 0.45 / 0.35
OOF NDCG@5: 0.91629
Балл: 91.629


In [23]:
# === Скачать результат локально на свой компьютер ===
import os, subprocess

FNAME = 'submission_v2.csv'
assert os.path.exists(FNAME), f'{FNAME} не найден — сначала выполни ячейку, которая его сохраняет'
print('Файл:', FNAME, '| размер:', round(os.path.getsize(FNAME) / 1e6, 2), 'МБ')

# Способ 1: нативное скачивание Colab.
# Работает в веб-Colab. Через VS Code-подключение к ядру Colab часто НЕ срабатывает
# (как и виджет upload) — тогда автоматически сработает Способ 2.
downloaded = False
try:
    from google.colab import files
    files.download(FNAME)
    print('OK: запущено скачивание через google.colab.files (проверь загрузки браузера)')
    downloaded = True
except Exception as e:
    print('google.colab.files недоступен здесь:', repr(e))

# Способ 2: заливаем на временный файлообменник и печатаем прямую ссылку.
# Открой ссылку в браузере НА СВОЁМ компьютере — файл скачается.
if not downloaded:
    out = subprocess.run(
        ['curl', '-s', '-A', 'Mozilla/5.0', '-F', f'file=@{FNAME}', 'https://0x0.st'],
        capture_output=True, text=True
    )
    url = out.stdout.strip()
    if url.startswith('http'):
        print('\n>>> Скачай файл, открыв эту ссылку в браузере на своём ПК:')
        print('   ', url)
    else:
        # запасной хостинг, если 0x0.st недоступен
        out2 = subprocess.run(
            ['curl', '-s', '-F', f'file=@{FNAME}', 'https://file.io'],
            capture_output=True, text=True
        )
        print('0x0.st не ответил. Ответ file.io (ссылка в поле "link"):')
        print(out2.stdout.strip())


AssertionError: submission_v2.csv не найден — сначала выполни ячейку, которая его сохраняет

In [24]:
test['pred_lgb'] = test_pred_lgb
test['pred_cb']  = test_pred_cb
test['pred_xgb'] = test_pred_xgb
test['r_lgb'] = rank_within_group(test, 'pred_lgb')
test['r_cb']  = rank_within_group(test, 'pred_cb')
test['r_xgb'] = rank_within_group(test, 'pred_xgb')

w_lgb, w_cb, w_xgb = best_w
test['score'] = w_lgb * test['r_lgb'] + w_cb * test['r_cb'] + w_xgb * test['r_xgb']

submission = test[['request_id', 'variant_no', 'score']].copy()
submission.to_csv('submission_v2.csv', index=False, sep=';')
print('shape:', submission.shape)
print(submission.head(10))
assert submission['score'].isna().sum() == 0
print('OK')

shape: (936883, 3)
   request_id  variant_no  score
0           0          10  0.100
1           0           5  0.700
2           0           7  0.395
3           0           6  0.495
4           0           8  0.355
5           0           3  0.900
6           0           4  0.800
7           0           9  0.295
8           0           2  0.460
9           0           1  1.000
OK


## Что ещё попробовать после v2

1. **Snapshot ensembling**: CatBoost с seeds [42,43,44,45,46], усреднить ранки → +0.05-0.15 п.п.
2. **CatBoost QuerySoftMax** (теоретически идеален для one-positive) как 4-я модель.
3. **Optuna** 50 trials для LGB и CB.
4. **Per-query z-score стэкинг** через LambdaMART meta-learner (паттерн Expedia 5-place) → +0.1-0.3 п.п.
